# 01 — Data preparation: Algerian dialect sentiment benchmark

Builds clean, leakage-free train/dev/test splits for three Algerian sentiment datasets and writes the dataset tables for the paper.

| Dataset | Source | Script | Classes | Licence |
|---|---|---|---|---|
| **TWIFIL** (Moudjari et al., LREC 2020) | X/Twitter, official split from the DziriBERT repo | Arabic + Latin, code-switched | 3 | CC BY-NC-SA 4.0 |
| **Algerian Dialect YouTube** (Benmounah et al., 2025) | YouTube comments, 30+ Algerian media channels | Arabic, some Latin | 5 → 3 | CC BY 4.0 |
| **NArabizi** (Touileb & Barnes, Findings ACL 2021) | Social media, Arabizi | Latin | 4 (3 + MIX) | see upstream repo |

**Before running:** download the YouTube dataset from Mendeley Data (DOI `10.17632/zzwg3nnhsz.2`) and put the file (zip, csv or xlsx, as downloaded) in `MyDrive/alg_sentiment_benchmark/raw/youtube/`. If it is missing, the notebook still prepares the other two datasets and tells you.

**Runtime:** CPU only, a few minutes. No GPU needed.

**Outputs** (all under `MyDrive/alg_sentiment_benchmark/`):
- `processed/<dataset>/{train,dev,test}.csv` with unified labels `0 = negative`, `1 = neutral`, `2 = positive` (NArabizi 4-class adds `3 = mixed`)
- `reports/dataset_stats.{csv,md}`: size, class balance, length, script, emoji share (Table: dataset statistics)
- `reports/cleaning_audit.{csv,md}`: what was removed and why (Table: data audit)
- `reports/provenance.json`: upstream commits, SHA-256 of raw files, config, library versions

## 1. Configuration

In [ ]:
# ---- Paths (Colab) ----
PROJECT_DIR = "/content/drive/MyDrive/alg_sentiment_benchmark"
YOUTUBE_RAW_DIR = f"{PROJECT_DIR}/raw/youtube"   # put the Mendeley download here

# ---- Reproducibility ----
SEED = 42
DEV_FRACTION = 0.10                 # dev carved (stratified) from train where no official dev exists
YOUTUBE_SPLIT = (0.80, 0.10, 0.10)  # train / dev / test for the YouTube dataset
YOUTUBE_GROUP_BY_VIDEO = False      # True: no video appears in two splits (stricter; fixes 80/10/10)
DROP_WITHIN_SPLIT_DUPLICATES = True # e.g. 54 identical "😂😂" tweets in TWIFIL train; False keeps them

# ---- YouTube column overrides (None = auto-detect; set these if detection picks the wrong column) ----
YT_TEXT_COL = None
YT_LABEL_COL = None
YT_VIDEO_COL = None

# YouTube labels: 0 very negative, 1 negative, 2 neutral, 3 positive, 4 very positive  ->  3 classes
YT_5_TO_3 = {0: 0, 1: 0, 2: 1, 3: 2, 4: 2}

# ---- Upstream repositories, pinned to the commits checked on 29 Sep 2026 ----
DZIRIBERT_REPO = ("https://github.com/alger-ia/dziribert.git", "8d6959ccec1cb9c475304ea97d0b68dff55c4377")
NARABIZI_REPO = ("https://github.com/SamiaTouileb/NArabizi.git", "1434f57bbaa718994f9080e4402f72d81a2962b2")

LABEL_NAMES = {0: "negative", 1: "neutral", 2: "positive", 3: "mixed"}

## 2. Environment

In [ ]:
import os, sys, re, html, json, glob, hashlib, zipfile, platform, subprocess, datetime

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    REPO_DIR = "/content/_repos"          # repos are re-cloned each session (fast, keeps Drive clean)
else:                                     # local run: paths overridable via environment variable
    PROJECT_DIR = os.environ.get("ALG_PROJECT_DIR", os.path.abspath("alg_sentiment_benchmark"))
    YOUTUBE_RAW_DIR = os.path.join(PROJECT_DIR, "raw", "youtube")
    REPO_DIR = os.path.join(PROJECT_DIR, "_repos")

RAW_DIR = os.path.join(PROJECT_DIR, "raw")
PROC_DIR = os.path.join(PROJECT_DIR, "processed")
REPORT_DIR = os.path.join(PROJECT_DIR, "reports")
for d in (RAW_DIR, YOUTUBE_RAW_DIR, PROC_DIR, REPORT_DIR, REPO_DIR):
    os.makedirs(d, exist_ok=True)

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print(f"Colab: {IN_COLAB} | Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print("Project dir:", PROJECT_DIR)

## 3. Fetch upstream repositories (pinned commits)

In [ ]:
def fetch_repo(url, commit):
    """Clone once, check out the pinned commit, return (path, actual HEAD)."""
    name = url.rstrip("/").split("/")[-1].replace(".git", "")
    path = os.path.join(REPO_DIR, name)
    if not os.path.isdir(os.path.join(path, ".git")):
        subprocess.run(["git", "clone", "-q", url, path], check=True)
    res = subprocess.run(["git", "-C", path, "checkout", "-q", commit], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"WARNING: pinned commit {commit[:10]} not found in {name}; using current HEAD")
    head = subprocess.run(["git", "-C", path, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
    print(f"{name:10s} @ {head[:10]}")
    return path, head

DZ_PATH, DZ_HEAD = fetch_repo(*DZIRIBERT_REPO)
NA_PATH, NA_HEAD = fetch_repo(*NARABIZI_REPO)

## 4. Text utilities

Cleaning is deliberately minimal and model-agnostic: URLs, @mentions, the `RT` marker and the `#` sign are removed; everything else (emojis, Arabizi digits, elongation, French words) is kept, because it carries sentiment and because each model family will apply its own preprocessing later.

A separate, aggressive **normalised key** (diacritics, alef/yaa/taa-marbuta variants, elongation, punctuation) is used **only** to detect duplicates, label conflicts and cross-split leakage.

In [ ]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+:?")
RT_RE = re.compile(r"^\s*RT\b\s*")
HASHTAG_RE = re.compile(r"#(\w+)")
ARABIC_LETTER_RE = re.compile(r"[ء-غف-يٱ-ۓۺ-ۿ]")
LATIN_LETTER_RE = re.compile(r"[A-Za-zÀ-ɏ]")
DIACRITICS_RE = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")
EMOJI_RE = re.compile(
    "["
    "\U0001F1E6-\U0001F1FF"   # flags
    "\U0001F300-\U0001F5FF"   # symbols & pictographs
    "\U0001F600-\U0001F64F"   # emoticons
    "\U0001F680-\U0001F6FF"   # transport & map
    "\U0001F700-\U0001FAFF"   # extended pictographs
    "☀-➿"           # misc symbols, dingbats (incl. heart)
    "]"
)
EMOTICON_RE = re.compile(r"(?:[:;=][-']?[()\[\]DPp/\\|]|<3)")
AR_NORM = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه", "ؤ": "و", "ئ": "ي", "ـ": ""})


def clean_text(text):
    s = html.unescape(str(text))
    s = RT_RE.sub(" ", s)
    s = URL_RE.sub(" ", s)
    s = MENTION_RE.sub(" ", s)
    s = HASHTAG_RE.sub(lambda m: m.group(1).replace("_", " "), s)
    return " ".join(s.split())


def has_content(text):
    """True if the cleaned text still has at least one letter, emoji or emoticon."""
    return bool(ARABIC_LETTER_RE.search(text) or LATIN_LETTER_RE.search(text)
                or EMOJI_RE.search(text) or EMOTICON_RE.search(text))


def dedup_key(text):
    """Aggressive normalisation used only for duplicate / conflict / leakage detection."""
    s = DIACRITICS_RE.sub("", text.lower()).translate(AR_NORM)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)                       # elongation: ههههه -> هه, cooool -> cool
    s = "".join(ch if (ch.isalnum() or ch.isspace() or EMOJI_RE.match(ch)) else " " for ch in s)
    s = " ".join(s.split())
    return s if s else text.strip()                              # emoticon-only texts keep their own key


def script_of(text):
    a, l = len(ARABIC_LETTER_RE.findall(text)), len(LATIN_LETTER_RE.findall(text))
    if a + l == 0:
        return "none"
    r = a / (a + l)
    return "arabic" if r >= 0.8 else ("latin" if r <= 0.2 else "mixed")


# quick self-test
assert clean_text("RT @user: رائع #الجزائر_الجديدة https://t.co/x") == "رائع الجزائر الجديدة"
assert not has_content(clean_text("@a @b https://t.co/x"))
assert has_content(clean_text("@a 😂")) and has_content(":)")
assert dedup_key("هههههه رائعة!!") == dedup_key("ههه رائعه")
assert script_of("sahit khouya") == "latin" and script_of("صحيت خويا") == "arabic"
print("text utilities OK")

## 5. Cleaning and audit engine

Applied in this order, each step logged per split:
1. **empty after URL/mention removal** — no letter, emoji or emoticon left (e.g. a tweet that is only `@user https://t.co/…`)
2. **conflicting labels** — the same normalised text appears with different labels: all copies dropped (label noise)
3. **duplicate within split** — first copy kept (switch off with `DROP_WITHIN_SPLIT_DUPLICATES = False`)
4. **cross-split leakage** — the same normalised text in two splits: the test copy is kept, then dev; the train copy is dropped, so official test sets change as little as possible

In [ ]:
AUDIT = []  # rows: dataset, step, split, n


def reset_audit(*datasets):
    """Makes each dataset section safe to re-run on its own."""
    AUDIT[:] = [r for r in AUDIT if r["dataset"] not in datasets]


def _log(dataset, step, series_split):
    for sp, n in series_split.value_counts().items():
        AUDIT.append({"dataset": dataset, "step": step, "split": sp, "n": int(n)})


def audit_clean(df, dataset):
    """df needs columns: id, split, text_raw, label. Returns the cleaned frame (with text, key)."""
    if len(df) == 0:
        raise ValueError(f"{dataset}: no rows to clean (check the loader above)")
    d = df.copy()
    d["_order"] = np.arange(len(d))
    d["text"] = d["text_raw"].map(clean_text)
    d["key"] = d["text"].map(dedup_key)
    _log(dataset, "0_input", d["split"])

    m = ~d["text"].map(has_content)
    _log(dataset, "1_empty_after_url_mention_removal", d.loc[m, "split"]); d = d[~m]

    m = d.groupby("key")["label"].transform("nunique") > 1
    _log(dataset, "2_conflicting_labels", d.loc[m, "split"]); d = d[~m]

    if DROP_WITHIN_SPLIT_DUPLICATES:
        m = d.duplicated(["split", "key"], keep="first")
        _log(dataset, "3_duplicate_within_split", d.loc[m, "split"]); d = d[~m]

    rank = d["split"].map({"test": 0, "dev": 1, "train": 2, "all": 3})
    d = d.assign(_rank=rank).sort_values(["key", "_rank", "_order"])
    m = d.duplicated("key", keep="first")
    _log(dataset, "4_cross_split_leakage", d.loc[m, "split"]); d = d[~m]

    d = d.sort_values("_order").drop(columns=["_order", "_rank"])
    _log(dataset, "5_output", d["split"])
    return d


def carve_dev(d, frac=DEV_FRACTION, seed=SEED, stratify_col="label"):
    """Move a stratified fraction of train into dev (used where no official dev split exists)."""
    d = d.copy()
    tr = d.index[d["split"] == "train"]
    _, dev_idx = train_test_split(tr, test_size=frac, stratify=d.loc[tr, stratify_col], random_state=seed)
    d.loc[dev_idx, "split"] = "dev"
    return d


def finalise(d, dataset):
    d = d.copy()
    d["dataset"] = dataset
    if "text" not in d:
        d["text"] = d["text_raw"]
    d["label"] = d["label"].astype(int)
    d["label_name"] = d["label"].map(LABEL_NAMES)
    stat_text = d["text"].map(clean_text)          # stats ignore URLs/mentions even for the raw official version
    d["script"] = stat_text.map(script_of)
    d["n_tokens"] = stat_text.str.split().str.len()
    cols = ["id", "dataset", "split", "text", "text_raw", "label", "label_name", "source_label", "script", "n_tokens"]
    if "video" in d:
        cols.append("video")
    return d[cols].reset_index(drop=True)


def save(d, name):
    out = os.path.join(PROC_DIR, name)
    os.makedirs(out, exist_ok=True)
    for sp in ("train", "dev", "test"):
        d[d["split"] == sp].to_csv(os.path.join(out, f"{sp}.csv"), index=False, encoding="utf-8")
    print(f"saved {name:22s}", d["split"].value_counts().reindex(["train", "dev", "test"]).to_dict())


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


RAW_FILES = {}   # path -> sha256, for provenance
DATASETS = {}    # name -> final frame

## 6. TWIFIL

Two versions are saved:
- **`twifil_official`** — the exact DziriBERT split (raw text, nothing removed), so results are comparable with the published numbers (DziriBERT 80.5 %, MARBERT 80.5 %, CAMeLBERT-mix 77.7 % accuracy). A stratified 10 % dev set is carved from train; merge it back if you need the full 7,077-tweet train set for strict replication.
- **`twifil_clean`** — the same split after the audit (empty, conflicting, duplicate and leaked tweets removed).

Label mapping verified against the original TWIFIL release: `0 = Negative`, `1 = Neutral`, `2 = Positive`.

In [ ]:
reset_audit("twifil_clean")
parts = []
for sp in ("train", "test"):
    path = os.path.join(DZ_PATH, "data", f"{sp}_sent.csv")
    RAW_FILES[path] = sha256(path)
    t = pd.read_csv(path).drop(columns=["Unnamed: 0"], errors="ignore").rename(columns={"text": "text_raw"})
    t["split"] = sp
    parts.append(t)
tw = pd.concat(parts, ignore_index=True)

n_nan = int(tw["text_raw"].isna().sum())
tw = tw.dropna(subset=["text_raw"])
tw["text_raw"] = tw["text_raw"].astype(str)
assert set(tw["label"].unique()) <= {0, 1, 2}, tw["label"].unique()
tw["source_label"] = tw["label"].map({0: "Negative", 1: "Neutral", 2: "Positive"})
tw["id"] = [f"twifil-{i:05d}" for i in range(len(tw))]
print(f"TWIFIL raw: {len(tw)} tweets ({n_nan} missing texts dropped)", tw["split"].value_counts().to_dict())

# official version: raw text, official test untouched
tw_off = carve_dev(tw)
DATASETS["twifil_official"] = finalise(tw_off, "twifil_official")
assert (DATASETS["twifil_official"]["split"] == "test").sum() == (tw["split"] == "test").sum()

# clean version
tw_clean = carve_dev(audit_clean(tw, "twifil_clean"))
DATASETS["twifil_clean"] = finalise(tw_clean, "twifil_clean")

## 7. NArabizi (Latin-script Arabizi)

Texts are read from the treebank `.conllu` files (`# text =` lines) and joined to the sentiment annotations by `sent_id`, keeping the official train/dev/test split. Two versions:
- **`narabizi`** — 3 classes; `MIX` sentences dropped (logged)
- **`narabizi_4class`** — all four labels (`3 = mixed`)

Small (≈1.3k sentences): use it as a **script-robustness test**, not as a main training set.

In [ ]:
SENT_ID_RE = re.compile(r"^#\s*sent_id\s*=\s*(.+)$")
TEXT_RE = re.compile(r"^#\s*text\s*=\s*(.*)$")


def read_conllu_texts(path):
    texts, sid = {}, None
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = SENT_ID_RE.match(line)
            if m:
                sid = m.group(1).strip()
                continue
            m = TEXT_RE.match(line)
            if m and sid is not None:
                texts[sid] = m.group(1).strip()
                sid = None
    return texts


reset_audit("narabizi", "narabizi_4class")
rows = []
for sp in ("train", "dev", "test"):
    conllu = os.path.join(NA_PATH, "data", "Narabizi", "pos", f"{sp}_NArabizi.conllu")
    sent = os.path.join(NA_PATH, "data", "Narabizi", "sentiment", f"{sp}_Narabizi_sentiment.txt")
    RAW_FILES[conllu], RAW_FILES[sent] = sha256(conllu), sha256(sent)
    texts = read_conllu_texts(conllu)
    ann = pd.read_csv(sent, sep="\t", dtype=str)
    ann.columns = ["sent_id", "source_label"]
    ann["sent_id"] = ann["sent_id"].str.replace(r"^#\s*sent_id\s*=\s*", "", regex=True).str.strip()  # IDs stored as '# sent_id = X'
    missing = ann[~ann["sent_id"].isin(texts)]
    if len(missing):
        print(f"WARNING {sp}: {len(missing)} annotated ids not found in conllu; dropped")
    ann = ann[ann["sent_id"].isin(texts)].copy()
    ann["text_raw"], ann["split"] = ann["sent_id"].map(texts), sp
    rows.append(ann)
na = pd.concat(rows, ignore_index=True)
na["source_label"] = na["source_label"].str.strip().str.upper()
assert set(na["source_label"]) <= {"NEG", "NEU", "POS", "MIX"}, set(na["source_label"])
na["id"] = "narabizi-" + na["split"] + "-" + na["sent_id"].astype(str)
print(f"NArabizi raw: {len(na)} sentences", na["split"].value_counts().to_dict(), na["source_label"].value_counts().to_dict())

na4 = na.assign(label=na["source_label"].map({"NEG": 0, "NEU": 1, "POS": 2, "MIX": 3}))
DATASETS["narabizi_4class"] = finalise(audit_clean(na4, "narabizi_4class"), "narabizi_4class")

mix = na4["label"] == 3
_log("narabizi", "0b_mixed_label_dropped", na4.loc[mix, "split"])
DATASETS["narabizi"] = finalise(audit_clean(na4[~mix], "narabizi"), "narabizi")

## 8. Algerian Dialect YouTube (Mendeley)

Auto-detects the file (zip, csv, tsv, xlsx, json) and the text / label / video columns, then prints them with examples of each label. **Check the printout:** label `0` must be *very negative* and `4` *very positive*. If a column is wrong, set `YT_TEXT_COL` / `YT_LABEL_COL` / `YT_VIDEO_COL` in section 1 and re-run.

The audit runs **before** splitting, so duplicates cannot leak across splits. The split is stratified on the original 5-class label, so the rare very-negative / very-positive comments are spread evenly.

In [ ]:
TABULAR_EXT = (".csv", ".tsv", ".txt", ".xlsx", ".xls", ".json", ".jsonl")
NAME_5 = {"very negative": 0, "negative": 1, "neutral": 2, "positive": 3, "very positive": 4,
          "très négatif": 0, "négatif": 1, "neutre": 2, "positif": 3, "très positif": 4}


def find_tabular_files(root):
    for z in glob.glob(os.path.join(root, "**", "*.zip"), recursive=True):
        out = os.path.join(root, "_extracted", os.path.splitext(os.path.basename(z))[0])
        if not os.path.isdir(out):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(out)
    files = [p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
             if p.lower().endswith(TABULAR_EXT) and "__MACOSX" not in p and os.path.isfile(p)]
    return sorted(files)


def read_any(path):
    p = path.lower()
    if p.endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    if p.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    if p.endswith(".json"):
        return pd.read_json(path)
    for enc in ("utf-8", "utf-8-sig", "cp1256"):
        for sep in (None, ",", "\t", ";"):
            try:
                df = pd.read_csv(path, sep=sep, engine="python", encoding=enc)
                if df.shape[1] >= 2:
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError, ValueError):
                continue
    raise ValueError(f"Could not parse {path}")


def _is_texty(s):
    v = s.dropna().astype(str).head(500)
    if len(v) == 0 or v.str.startswith("http").mean() > 0.5:
        return False
    letters = v.map(lambda t: len(ARABIC_LETTER_RE.findall(t)) + len(LATIN_LETTER_RE.findall(t)))
    return (letters >= 3).mean() > 0.8


def _is_label_like(s):
    v = s.dropna()
    if len(v) == 0:
        return False
    num = pd.to_numeric(v, errors="coerce")
    if num.notna().mean() > 0.99:
        vals = set(num.dropna().astype(int).unique())
        return vals <= set(range(5)) and len(vals) >= 3
    return v.astype(str).str.strip().str.lower().isin(NAME_5).mean() > 0.99


def detect_columns(df):
    cols = list(df.columns)
    low = {c: str(c).lower() for c in cols}

    def by_name(keys, cond):
        for k in keys:
            for c in cols:
                if k in low[c] and cond(df[c]):
                    return c
        return None

    text = YT_TEXT_COL or by_name(["comment", "text", "content", "post"], _is_texty)
    if text is None:
        texty = [c for c in cols if _is_texty(df[c])]
        text = max(texty, key=lambda c: df[c].astype(str).str.len().median(), default=None)
    label = YT_LABEL_COL or by_name(["label", "sentiment", "polarity", "class", "score"], _is_label_like)
    if label is None:
        label = next((c for c in cols if _is_label_like(df[c])), None)
    video = YT_VIDEO_COL or by_name(["video", "url", "link"], lambda s: True)
    return text, label, video


reset_audit("youtube")
DATASETS.pop("youtube", None)
yt_files = find_tabular_files(YOUTUBE_RAW_DIR)
if not yt_files:
    print(f"YouTube dataset not found in {YOUTUBE_RAW_DIR}\n"
          "-> download it from Mendeley Data (DOI 10.17632/zzwg3nnhsz.2), put the file there and re-run from section 8.\n"
          "   Continuing with TWIFIL and NArabizi only.")
else:
    # pick the largest parsable table that has detectable text and label columns
    candidates = []
    for f in yt_files:
        try:
            df = read_any(f)
        except Exception as e:
            print(f"skip {os.path.basename(f)}: {e}")
            continue
        tcol, lcol, vcol = detect_columns(df)
        print(f"{os.path.basename(f)}: {df.shape}, text={tcol!r}, label={lcol!r}, video={vcol!r}")
        if tcol is not None and lcol is not None:
            candidates.append((len(df), f, df, tcol, lcol, vcol))
    if not candidates:
        raise ValueError("No file with detectable text and label columns. Set YT_TEXT_COL / YT_LABEL_COL in section 1. "
                         f"Columns seen: {[list(read_any(f).columns) for f in yt_files]}")
    _, yt_file, yt_raw, TCOL, LCOL, VCOL = max(candidates, key=lambda c: c[0])
    RAW_FILES[yt_file] = sha256(yt_file)
    print(f"\nUsing {yt_file}\n  text column : {TCOL!r}\n  label column: {LCOL!r}\n  video column: {VCOL!r}")

    yt = pd.DataFrame({"text_raw": yt_raw[TCOL], "lab": yt_raw[LCOL]})
    if VCOL is not None:
        # URLs point to single comments (...watch?v=<video>&lc=<comment>): keep the video id only;
        # rows without a URL get their own group so they never form one giant "unknown" video
        vid = yt_raw[VCOL].astype(str).str.extract(r"(?:[?&]v=|youtu\.be/|/shorts/)([\w-]{6,})")[0]
        yt["video"] = [v if isinstance(v, str) else f"novideo-{i}" for i, v in enumerate(vid)]
        print(f"  videos: {vid.nunique()} distinct ids, {int(vid.isna().sum())} comments without a URL")
    n0 = len(yt)
    yt = yt.dropna(subset=["text_raw", "lab"])
    num = pd.to_numeric(yt["lab"], errors="coerce")
    if num.notna().all():
        yt["lab5"] = num.astype(int)
    else:
        yt["lab5"] = yt["lab"].astype(str).str.strip().str.lower().map(NAME_5)
        yt = yt.dropna(subset=["lab5"]).astype({"lab5": int})
    print(f"  rows: {n0} read, {len(yt)} with text and label")
    assert set(yt["lab5"].unique()) <= set(range(5)), yt["lab5"].unique()

    # --- manual check of label order ---
    print("\nLabel distribution (5-class):", yt["lab5"].value_counts().sort_index().to_dict())
    for v in sorted(yt["lab5"].unique()):
        ex = yt.loc[yt["lab5"] == v, "text_raw"].sample(min(2, (yt["lab5"] == v).sum()), random_state=SEED)
        for t in ex:
            print(f"  [{v}] {str(t)[:110]}")

    yt["text_raw"] = yt["text_raw"].astype(str)
    yt["source_label"] = yt["lab5"].astype(str)
    yt["label"] = yt["lab5"].map(YT_5_TO_3)
    yt["split"] = "all"
    yt["id"] = [f"yt-{i:05d}" for i in range(len(yt))]
    yt = audit_clean(yt, "youtube").reset_index(drop=True)

    # --- split after cleaning (no leakage by construction) ---
    grouped = YOUTUBE_GROUP_BY_VIDEO and "video" in yt and yt["video"].nunique() > 20
    if YOUTUBE_GROUP_BY_VIDEO and not grouped:
        print("WARNING: video column missing or too few videos; falling back to a stratified random split")
    if grouped:
        sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
        folds = list(sgkf.split(yt, yt["source_label"], groups=yt["video"]))
        yt["split"] = "train"
        yt.loc[folds[0][1], "split"] = "test"
        yt.loc[folds[1][1], "split"] = "dev"
    else:
        tr, te = train_test_split(yt.index, test_size=YOUTUBE_SPLIT[2], stratify=yt["source_label"], random_state=SEED)
        dev_share = YOUTUBE_SPLIT[1] / (YOUTUBE_SPLIT[0] + YOUTUBE_SPLIT[1])
        tr, dv = train_test_split(tr, test_size=dev_share, stratify=yt.loc[tr, "source_label"], random_state=SEED)
        yt["split"] = "train"
        yt.loc[te, "split"] = "test"
        yt.loc[dv, "split"] = "dev"
    _log("youtube", "6_split", yt["split"])
    DATASETS["youtube"] = finalise(yt, "youtube")
    print("\nYouTube split:", DATASETS["youtube"]["split"].value_counts().to_dict(), "| grouped by video:", grouped)

## 9. Sanity checks and save

In [ ]:
for name, d in DATASETS.items():
    assert d["text"].str.strip().str.len().gt(0).all() or name == "twifil_official", name
    assert set(d["label"].unique()) <= set(LABEL_NAMES), name
    assert set(d["split"].unique()) == {"train", "dev", "test"}, (name, d["split"].unique())
    if name != "twifil_official":                                   # official split is kept as released
        keys = d.assign(k=d["text"].map(dedup_key)).groupby("k")["split"].nunique()
        assert (keys == 1).all(), f"{name}: {int((keys > 1).sum())} texts shared across splits"
        assert d["text"].map(has_content).all(), name
print("sanity checks passed\n")

for name, d in DATASETS.items():
    save(d, name)

## 10. Tables for the paper

In [ ]:
def df_to_md(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r.values) + " |")
    return "\n".join(lines) + "\n"


# ---- dataset statistics ----
rows = []
for name, d in DATASETS.items():
    labels_present = sorted(d["label"].unique())
    for sp in ("train", "dev", "test", "all"):
        s = d if sp == "all" else d[d["split"] == sp]
        r = {"dataset": name, "split": sp, "n": len(s)}
        for lab in labels_present:
            r[f"% {LABEL_NAMES[lab][:3]}"] = round(100 * (s["label"] == lab).mean(), 1)
        r["median tokens"] = float(s["n_tokens"].median())
        r["mean tokens"] = round(s["n_tokens"].mean(), 1)
        for sc in ("arabic", "latin", "mixed"):
            r[f"% {sc.capitalize()} script"] = round(100 * (s["script"] == sc).mean(), 1)
        r["% with emoji"] = round(100 * s["text"].map(lambda t: bool(EMOJI_RE.search(t))).mean(), 1)
        r["% no text (URL/mention only)"] = round(100 * (~s["text"].map(clean_text).map(has_content)).mean(), 1)
        rows.append(r)
stats = pd.DataFrame(rows)
lab_cols = [c for c in ("% neg", "% neu", "% pos", "% mix") if c in stats.columns]
stats = stats[["dataset", "split", "n"] + lab_cols + [c for c in stats.columns if c not in lab_cols + ["dataset", "split", "n"]]]
stats.to_csv(os.path.join(REPORT_DIR, "dataset_stats.csv"), index=False)
with open(os.path.join(REPORT_DIR, "dataset_stats.md"), "w", encoding="utf-8") as f:
    f.write(df_to_md(stats))
print("DATASET STATISTICS")
print(stats.to_string(index=False))

# ---- cleaning audit ----
audit = pd.DataFrame(AUDIT)
audit = (audit.pivot_table(index=["dataset", "step"], columns="split", values="n", aggfunc="sum", fill_value=0)
              .reset_index())
audit.columns.name = None
split_cols = [c for c in ("all", "train", "dev", "test") if c in audit.columns]
audit = audit[["dataset", "step"] + split_cols]
audit["total"] = audit[split_cols].sum(axis=1)
audit.to_csv(os.path.join(REPORT_DIR, "cleaning_audit.csv"), index=False)
with open(os.path.join(REPORT_DIR, "cleaning_audit.md"), "w", encoding="utf-8") as f:
    f.write(df_to_md(audit))
print("\nCLEANING AUDIT (rows removed per step; 0_input / 5_output are counts)")
print(audit.to_string(index=False))

# ---- leakage in the official TWIFIL split, for the paper's data section ----
off = DATASETS["twifil_official"]
k = off.assign(c=off["text_raw"].map(clean_text))
k = k.assign(has=k["c"].map(has_content), k=k["c"].map(dedup_key))
tr_keys = set(k.loc[(k["split"] != "test") & k["has"], "k"])
te = k[k["split"] == "test"]
n_empty, n_leak = int((~te["has"]).sum()), int((te["has"] & te["k"].isin(tr_keys)).sum())
emp = k[~k["has"]]
print(f"\nOfficial TWIFIL, all splits: {len(emp)} tweets have no text after removing mentions/URLs; label distribution "
      f"{emp['label_name'].value_counts().to_dict()} -> a 'URL-only = neutral' shortcut inflates official scores")
print(f"\nOfficial TWIFIL test ({len(te)} tweets): {n_empty} ({100*n_empty/len(te):.1f}%) have no text after removing "
      f"mentions/URLs; {n_leak} ({100*n_leak/len(te):.1f}%) of the rest also occur in train/dev after normalisation")

## 11. Provenance

In [ ]:
prov = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "repos": {"dziribert": DZ_HEAD, "NArabizi": NA_HEAD},
    "raw_files_sha256": {os.path.relpath(p, REPO_DIR if p.startswith(REPO_DIR) else PROJECT_DIR): h
                         for p, h in RAW_FILES.items()},
    "config": {"SEED": SEED, "DEV_FRACTION": DEV_FRACTION, "YOUTUBE_SPLIT": YOUTUBE_SPLIT,
               "YOUTUBE_GROUP_BY_VIDEO": YOUTUBE_GROUP_BY_VIDEO, "DROP_WITHIN_SPLIT_DUPLICATES": DROP_WITHIN_SPLIT_DUPLICATES,
               "YT_5_TO_3": YT_5_TO_3},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                 "scikit-learn": sklearn.__version__},
    "counts": {n: d["split"].value_counts().to_dict() for n, d in DATASETS.items()},
}
with open(os.path.join(REPORT_DIR, "provenance.json"), "w", encoding="utf-8") as f:
    json.dump(prov, f, indent=2, ensure_ascii=False)
print(json.dumps(prov["counts"], indent=2))
print("\nDone. Files in", PROJECT_DIR)

## Notes for the paper

- Report **both** TWIFIL versions: `twifil_official` for comparability with published results, `twifil_clean` as the main benchmark. The audit table is the justification.
- The processed CSVs keep `text` (minimally cleaned) and `text_raw`. Model-specific preprocessing (Arabic normalisation for TF-IDF, tokenisers for BERTs) belongs in the modelling notebook, not here.
- Do not redistribute the original TWIFIL user metadata (names, screen names); these files contain text and labels only. TWIFIL is CC BY-NC-SA 4.0, so any release of derived splits must use the same licence.
- Next notebook: `02_models_and_benchmark.ipynb` (training, ONNX export, int8 quantisation, latency/memory/size measurement, Pareto plots).